# S3 direto, sem catálogo no AWS Glue 6.0

Este notebook é um **teste**: ele lê um CSV pequeno da `landing` e grava/relê os dados **direto por caminho** em `s3://notislake/bronze/...`, **sem registrar nada no catálogo**. A Parte A usa a API do Spark nos formatos Iceberg, Delta Lake, Hudi, Parquet, CSV e ORC; a Parte B usa **DynamicFrame** (`from_options`) nos formatos que ele suporta. Todas as células podem ser reexecutadas e, se alguma falhar, a falha é um resultado válido do teste: ela também aconteceria na AWS.

## 1. Sessão

O runtime **AWS Glue 6.0** já configura extensões e acesso ao `s3`; só precisamos obter a sessão a partir do `GlueContext`.

In [ ]:
from awsglue.context import GlueContext
from pyspark.context import SparkContext

glueContext = GlueContext(SparkContext.getOrCreate())
spark = glueContext.spark_session

## 2. Preparar a origem

Gravamos um CSV **com cabeçalho** em `landing/teste_glue/pessoas/`. Este mesmo arquivo é a origem de outros notebooks de teste. `overwrite` mantém a célula reexecutável.

In [ ]:
ORIGEM = "s3://notislake/landing/teste_glue/pessoas/"
schema = "id INT, nome STRING, area STRING, valor DOUBLE, dt STRING"

pessoas = spark.createDataFrame(
    [
        (1, "Ana", "Dados", 100.5, "2026-01-01"),
        (2, "Bruno", "Plataforma", 200.0, "2026-01-02"),
        (3, "Carla", "Segurança", 300.25, "2026-01-03"),
        (4, "Diego", "Dados", 400.0, "2026-01-04"),
        (5, "Elisa", "Plataforma", 500.75, "2026-01-05"),
    ],
    schema,
)

pessoas.coalesce(1).write.mode("overwrite").option("header", True).csv(ORIGEM)

## 3. Ler o CSV

Lemos o CSV com **schema explícito** (o cabeçalho é apenas descartado) e guardamos o resultado para as duas partes.

In [ ]:
df = spark.read.schema(schema).option("header", True).csv(ORIGEM)
df.orderBy("id").show()
print(df.count())

# Parte A. API Spark (`DataFrame`)

Em cada formato gravamos com `.save(<caminho>)` e relemos do **mesmo caminho**. Como é só por caminho, nada aparece no catálogo. A contagem de linhas esperada é **5**.

## 4. Iceberg por caminho

Atenção: o Iceberg foi pensado para trabalhar com **catálogo** (por exemplo `glue_catalog.<banco>.<tabela>`). Gravar por caminho, sem catálogo, usa o `HadoopTables` do Iceberg e **pode não funcionar** no Glue. Esta célula é um teste: se falhar aqui, a mesma mensagem aparece na AWS, e o resultado é válido. Para Iceberg com catálogo, veja o sample `iceberg`.

In [ ]:
destino_iceberg = "s3://notislake/bronze/teste_s3_iceberg/"

df.write.format("iceberg").mode("overwrite").save(destino_iceberg)

In [ ]:
lido_iceberg = spark.read.format("iceberg").load(destino_iceberg)
lido_iceberg.orderBy("id").show()
print(lido_iceberg.count())

## 5. Delta Lake por caminho

In [ ]:
destino_delta = "s3://notislake/bronze/teste_s3_delta/"

df.write.format("delta").mode("overwrite").save(destino_delta)

In [ ]:
lido_delta = spark.read.format("delta").load(destino_delta)
lido_delta.orderBy("id").show()
print(lido_delta.count())

## 6. Hudi por caminho

O Hudi exige o nome da tabela (`hoodie.table.name`), a chave do registro (`id`) e o campo de desempate (`dt`). Por caminho, ele **não sincroniza** com o catálogo.

In [ ]:
destino_hudi = "s3://notislake/bronze/teste_s3_hudi/"

hudi_options = {
    "hoodie.table.name": "teste_s3_hudi",
    "hoodie.datasource.write.recordkey.field": "id",
    "hoodie.datasource.write.precombine.field": "dt",
}

df.write.format("hudi").options(**hudi_options).mode("overwrite").save(destino_hudi)

In [ ]:
lido_hudi = spark.read.format("hudi").load(destino_hudi)
lido_hudi.select("id", "nome", "area", "valor", "dt").orderBy("id").show()
print(lido_hudi.count())

## 7. Parquet, CSV e ORC por caminho

Formatos de arquivo simples: `overwrite` substitui o conteúdo da pasta a cada execução.

In [ ]:
destino_parquet = "s3://notislake/bronze/teste_s3_parquet/"

df.write.mode("overwrite").parquet(destino_parquet)

In [ ]:
lido_parquet = spark.read.parquet(destino_parquet)
lido_parquet.orderBy("id").show()
print(lido_parquet.count())

In [ ]:
destino_csv = "s3://notislake/bronze/teste_s3_csv/"

df.write.mode("overwrite").option("header", True).csv(destino_csv)

In [ ]:
lido_csv = spark.read.schema(schema).option("header", True).csv(destino_csv)
lido_csv.orderBy("id").show()
print(lido_csv.count())

In [ ]:
destino_orc = "s3://notislake/bronze/teste_s3_orc/"

df.write.mode("overwrite").orc(destino_orc)

In [ ]:
lido_orc = spark.read.orc(destino_orc)
lido_orc.orderBy("id").show()
print(lido_orc.count())

# Parte B. DynamicFrame (`from_options`)

O `DynamicFrame` lê e grava no `s3` com `connection_type="s3"` e `format=...`. Ele suporta **csv, json, parquet, glueparquet e orc**. Delta Lake, Hudi e Iceberg **não** são lidos nem gravados por `DynamicFrame`: nesses formatos o Glue usa a API Spark da Parte A.

A gravação do `DynamicFrame` **acrescenta** arquivos ao destino; por isso apagamos a pasta antes de gravar, para reexecutar sem duplicar linhas.

## 8. Ler o CSV como DynamicFrame

Lemos o mesmo CSV da `landing`, informando que a primeira linha é o cabeçalho.

In [ ]:
dyf = glueContext.create_dynamic_frame.from_options(
    connection_type="s3",
    connection_options={"paths": [ORIGEM]},
    format="csv",
    format_options={"withHeader": True},
)
dyf.toDF().orderBy("id").show()
print(dyf.count())

## 9. Função de apoio: limpar o destino

Apagamos a pasta de destino pela API de sistema de arquivos do Hadoop (a mesma que o Spark usa para o `s3://`). Se a pasta não existir, nada acontece.

In [ ]:
def limpar(caminho):
    jvm = spark._jvm
    hadoop_path = jvm.org.apache.hadoop.fs.Path(caminho)
    fs = hadoop_path.getFileSystem(spark._jsc.hadoopConfiguration())
    return fs.delete(hadoop_path, True)

## 10. Gravar e reler em cada formato

Para cada formato: limpamos o destino, gravamos com `write_dynamic_frame.from_options` e relemos com `create_dynamic_frame.from_options`. O CSV é gravado com cabeçalho e relido com `withHeader`.

In [ ]:
destino_dyf_csv = "s3://notislake/bronze/teste_dyf_csv/"

limpar(destino_dyf_csv)
glueContext.write_dynamic_frame.from_options(
    frame=dyf,
    connection_type="s3",
    connection_options={"path": destino_dyf_csv},
    format="csv",
    format_options={"writeHeader": True},
)

In [ ]:
lido_dyf_csv = glueContext.create_dynamic_frame.from_options(
    connection_type="s3",
    connection_options={"paths": [destino_dyf_csv]},
    format="csv",
    format_options={"withHeader": True},
)
lido_dyf_csv.toDF().orderBy("id").show()
print(lido_dyf_csv.count())

In [ ]:
destino_dyf_parquet = "s3://notislake/bronze/teste_dyf_parquet/"

limpar(destino_dyf_parquet)
glueContext.write_dynamic_frame.from_options(
    frame=dyf,
    connection_type="s3",
    connection_options={"path": destino_dyf_parquet},
    format="parquet",
)

In [ ]:
lido_dyf_parquet = glueContext.create_dynamic_frame.from_options(
    connection_type="s3",
    connection_options={"paths": [destino_dyf_parquet]},
    format="parquet",
)
lido_dyf_parquet.toDF().orderBy("id").show()
print(lido_dyf_parquet.count())

In [ ]:
destino_dyf_glueparquet = "s3://notislake/bronze/teste_dyf_glueparquet/"

limpar(destino_dyf_glueparquet)
glueContext.write_dynamic_frame.from_options(
    frame=dyf,
    connection_type="s3",
    connection_options={"path": destino_dyf_glueparquet},
    format="glueparquet",
    format_options={"compression": "snappy"},
)

In [ ]:
lido_dyf_glueparquet = glueContext.create_dynamic_frame.from_options(
    connection_type="s3",
    connection_options={"paths": [destino_dyf_glueparquet]},
    format="parquet",
)
lido_dyf_glueparquet.toDF().orderBy("id").show()
print(lido_dyf_glueparquet.count())

In [ ]:
destino_dyf_orc = "s3://notislake/bronze/teste_dyf_orc/"

limpar(destino_dyf_orc)
glueContext.write_dynamic_frame.from_options(
    frame=dyf,
    connection_type="s3",
    connection_options={"path": destino_dyf_orc},
    format="orc",
)

In [ ]:
lido_dyf_orc = glueContext.create_dynamic_frame.from_options(
    connection_type="s3",
    connection_options={"paths": [destino_dyf_orc]},
    format="orc",
)
lido_dyf_orc.toDF().orderBy("id").show()
print(lido_dyf_orc.count())

In [ ]:
destino_dyf_json = "s3://notislake/bronze/teste_dyf_json/"

limpar(destino_dyf_json)
glueContext.write_dynamic_frame.from_options(
    frame=dyf,
    connection_type="s3",
    connection_options={"path": destino_dyf_json},
    format="json",
)

In [ ]:
lido_dyf_json = glueContext.create_dynamic_frame.from_options(
    connection_type="s3",
    connection_options={"paths": [destino_dyf_json]},
    format="json",
)
lido_dyf_json.toDF().orderBy("id").show()
print(lido_dyf_json.count())

## 11. Delta Lake, Hudi e Iceberg com DynamicFrame

O Glue **não** usa `DynamicFrame` para ler ou gravar tabelas Delta Lake, Hudi e Iceberg; nesses formatos usamos a API Spark da Parte A (`df.write.format(...).save(...)`).

## 12. Resumo

Relemos **todos** os destinos e montamos uma tabela `formato | caminho | linhas lidas`. Cada formato é lido em um `try` próprio: se uma gravação anterior falhou, o resumo mostra `ERRO` em vez de interromper o notebook. O esperado é **5** linhas em cada formato.

In [ ]:
leituras = [
    ("iceberg (Spark)", destino_iceberg, lambda c: spark.read.format("iceberg").load(c)),
    ("delta (Spark)", destino_delta, lambda c: spark.read.format("delta").load(c)),
    ("hudi (Spark)", destino_hudi, lambda c: spark.read.format("hudi").load(c)),
    ("parquet (Spark)", destino_parquet, lambda c: spark.read.parquet(c)),
    ("csv (Spark)", destino_csv, lambda c: spark.read.schema(schema).option("header", True).csv(c)),
    ("orc (Spark)", destino_orc, lambda c: spark.read.orc(c)),
    ("csv (DynamicFrame)", destino_dyf_csv, lambda c: glueContext.create_dynamic_frame.from_options(
        connection_type="s3", connection_options={"paths": [c]}, format="csv",
        format_options={"withHeader": True}).toDF()),
    ("parquet (DynamicFrame)", destino_dyf_parquet, lambda c: glueContext.create_dynamic_frame.from_options(
        connection_type="s3", connection_options={"paths": [c]}, format="parquet").toDF()),
    ("glueparquet (DynamicFrame)", destino_dyf_glueparquet, lambda c: glueContext.create_dynamic_frame.from_options(
        connection_type="s3", connection_options={"paths": [c]}, format="parquet").toDF()),
    ("orc (DynamicFrame)", destino_dyf_orc, lambda c: glueContext.create_dynamic_frame.from_options(
        connection_type="s3", connection_options={"paths": [c]}, format="orc").toDF()),
    ("json (DynamicFrame)", destino_dyf_json, lambda c: glueContext.create_dynamic_frame.from_options(
        connection_type="s3", connection_options={"paths": [c]}, format="json").toDF()),
]

resumo = []
for formato, caminho, ler in leituras:
    try:
        linhas = str(ler(caminho).count())
    except Exception as erro:
        linhas = "ERRO: " + type(erro).__name__
    resumo.append((formato, caminho, linhas))

spark.createDataFrame(resumo, "formato STRING, caminho STRING, linhas_lidas STRING").show(truncate=False)